# Lab: Data Structuring and Combining Data

This notebook completes:

# Challenge 1 — Combining & Cleaning Data  
Using file1.csv, file2.csv, file3.csv

# Challenge 2 — Structuring Data  
Using marketing_customer_analysis_clean.csv

We will:

- Load and clean all datasets  
- Combine them  
- Apply formatting  
- Create pivot tables  
- Reshape data  
- Analyze marketing and customer behavior  


In [1]:
# 1. Import core libraries
import pandas as pd
import numpy as np

# 2. Display options
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', lambda x: f'{x:,.2f}')


In [2]:
# 2. Load the cleaned dataset created in the previous lab
df_file1 = pd.read_csv("cleaned_customer_analysis.csv")

df_file1.head()

,customer,state,gender,education,customer_lifetime_value,income,monthly_premium_auto,number_of_open_complaints,policy_type,vehicle_class,total_claim_amount
0,RB50392,Washington,Unknown,Master,793690,0,1000,0,Personal Auto,Four-Door Car,2.70
1,QZ44356,Arizona,F,Bachelor,697953,0,94,0,Personal Auto,Four-Door Car,"1,131.46"
2,AI49188,Nevada,F,Bachelor,1288743,48767,108,0,Personal Auto,Two-Door Car,566.47
3,WW63253,California,M,Bachelor,764586,0,106,0,Corporate Auto,SUV,529.88
4,GA49547,Washington,M,High School or Below,536307,36357,68,0,Personal Auto,Four-Door Car,17.27


# Inspecting the cleaned dataset

First check the structure, 
data types,
and basic statistics to understand what we are working with.


In [3]:
# 3 — Inspect the Dataset Structure

df_file1.info()
df_file1.describe()
df_file1.columns

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1072 entries, 0 to 1071
Data columns (total 11 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   customer                   1071 non-null   object 
 1   state                      1072 non-null   object 
 2   gender                     1072 non-null   object 
 3   education                  1071 non-null   object 
 4   customer_lifetime_value    1072 non-null   int64  
 5   income                     1072 non-null   int64  
 6   monthly_premium_auto       1072 non-null   int64  
 7   number_of_open_complaints  1072 non-null   int64  
 8   policy_type                1071 non-null   object 
 9   vehicle_class              1071 non-null   object 
 10  total_claim_amount         1071 non-null   float64
dtypes: float64(1), int64(4), object(6)
memory usage: 92.3+ KB


Index(['customer', 'state', 'gender', 'education', 'customer_lifetime_value',
       'income', 'monthly_premium_auto', 'number_of_open_complaints',
       'policy_type', 'vehicle_class', 'total_claim_amount'],
      dtype='object')

In [4]:
# Load file2 
df_file2 = pd.read_csv("file2.csv")

# Load file3
df_file3 = pd.read_csv("file3.csv")


In [5]:
# check file2 to see the dataset
df_file2.head()

,Customer,ST,GENDER,Education,Customer Lifetime Value,Income,Monthly Premium Auto,Number of Open Complaints,Total Claim Amount,Policy Type,Vehicle Class
0,GS98873,Arizona,F,Bachelor,323912.47%,16061,88,1/0/00,633.60,Personal Auto,Four-Door Car
1,CW49887,California,F,Master,462680.11%,79487,114,1/0/00,547.20,Special Auto,SUV
2,MY31220,California,F,College,899704.02%,54230,112,1/0/00,537.60,Personal Auto,Two-Door Car
3,UH35128,Oregon,F,College,2580706.30%,71210,214,1/1/00,"1,027.20",Personal Auto,Luxury Car
4,WH52799,Arizona,F,College,380812.21%,94903,94,1/0/00,451.20,Corporate Auto,Two-Door Car


In [6]:
# check file3 to see the dataset
df_file3.head()

,Customer,State,Customer Lifetime Value,Education,Gender,Income,Monthly Premium Auto,Number of Open Complaints,Policy Type,Total Claim Amount,Vehicle Class
0,SA25987,Washington,"3,479.14",High School or Below,M,0,104,0,Personal Auto,499.20,Two-Door Car
1,TB86706,Arizona,"2,502.64",Master,M,0,66,0,Personal Auto,3.47,Two-Door Car
2,ZL73902,Nevada,"3,265.16",Bachelor,F,25820,82,0,Personal Auto,393.60,Four-Door Car
3,KX23516,California,"4,455.84",High School or Below,F,0,121,0,Personal Auto,699.62,SUV
4,FN77294,California,"7,704.96",High School or Below,M,30366,101,2,Personal Auto,484.80,SUV


## Challenge 1 — Cleaning file2 and file3

We apply the same cleaning rules used in the previous lab:

- Standardize column names  
- Fix gender inconsistencies  
- Replace state abbreviations  
- Clean education values  
- Remove % from CLV  
- Fix number_of_open_complaints  
- Convert numeric columns  


In [7]:
# Cleaning function reused from previous lab
def clean_data(df):
    # Standardize column names
    df.columns = df.columns.str.lower().str.replace(" ", "_")
    df = df.rename(columns={"st": "state"})
    
    # Gender cleanup
    gender_map = {
        "female": "F", "Femal": "F", "f": "F", "F": "F",
        "male": "M", "m": "M", "Male": "M", "M": "M"
    }
    df["gender"] = df["gender"].astype(str).map(gender_map).fillna("Unknown")
    
    # State cleanup
    state_map = {"AZ": "Arizona", "Cali": "California", "WA": "Washington"}
    df["state"] = df["state"].replace(state_map)
    
    # Education cleanup
    df["education"] = df["education"].replace("Bachelors", "Bachelor")
    
    # CLV cleanup
    df["customer_lifetime_value"] = (
        df["customer_lifetime_value"]
        .astype(str)
        .str.replace("%", "")
    )
    df["customer_lifetime_value"] = pd.to_numeric(df["customer_lifetime_value"], errors="coerce")
    
    # Complaints cleanup
    df["number_of_open_complaints"] = (
        df["number_of_open_complaints"]
        .astype(str)
        .str.split("/")
        .str[1]
    )
    df["number_of_open_complaints"] = pd.to_numeric(df["number_of_open_complaints"], errors="coerce")
    
    # Fill nulls
    df["gender"] = df["gender"].fillna("Unknown")
    df["state"] = df["state"].fillna("Unknown")
    df["customer_lifetime_value"] = df["customer_lifetime_value"].fillna(df["customer_lifetime_value"].mean())
    df["number_of_open_complaints"] = df["number_of_open_complaints"].fillna(0)
    df["income"] = df["income"].fillna(0)
    df["monthly_premium_auto"] = df["monthly_premium_auto"].fillna(0)
    
    # Convert numeric columns
    df["income"] = df["income"].astype(int)
    df["monthly_premium_auto"] = df["monthly_premium_auto"].astype(int)
    df["number_of_open_complaints"] = df["number_of_open_complaints"].astype(int)
    df["customer_lifetime_value"] = df["customer_lifetime_value"].astype(int)
    
    # Remove duplicates
    df = df.drop_duplicates().reset_index(drop=True)
    
    return df


In [8]:
# Clean file2 and file3
clean_file2 = clean_data(df_file2)
clean_file3 = clean_data(df_file3)

# Display file2
clean_file2.head()

,customer,state,gender,education,customer_lifetime_value,income,monthly_premium_auto,number_of_open_complaints,total_claim_amount,policy_type,vehicle_class
0,GS98873,Arizona,F,Bachelor,323912,16061,88,0,633.60,Personal Auto,Four-Door Car
1,CW49887,California,F,Master,462680,79487,114,0,547.20,Special Auto,SUV
2,MY31220,California,F,College,899704,54230,112,0,537.60,Personal Auto,Two-Door Car
3,UH35128,Oregon,F,College,2580706,71210,214,1,"1,027.20",Personal Auto,Luxury Car
4,WH52799,Arizona,F,College,380812,94903,94,0,451.20,Corporate Auto,Two-Door Car


In [9]:
# Display file3
clean_file3.head()

,customer,state,customer_lifetime_value,education,gender,income,monthly_premium_auto,number_of_open_complaints,policy_type,total_claim_amount,vehicle_class
0,SA25987,Washington,3479,High School or Below,M,0,104,0,Personal Auto,499.20,Two-Door Car
1,TB86706,Arizona,2502,Master,M,0,66,0,Personal Auto,3.47,Two-Door Car
2,ZL73902,Nevada,3265,Bachelor,F,25820,82,0,Personal Auto,393.60,Four-Door Car
3,KX23516,California,4455,High School or Below,F,0,121,0,Personal Auto,699.62,SUV
4,FN77294,California,7704,High School or Below,M,30366,101,0,Personal Auto,484.80,SUV


# Combine all three cleaned datasets

We now concatenate:

- cleaned file1  
- cleaned file2  
- cleaned file3  

into one unified dataset.


In [10]:
# combine all three dataset
combined_df = pd.concat([df_file1, clean_file2, clean_file3], axis=0, ignore_index=True)

display(combined_df.head())
print(combined_df.shape)


,customer,state,gender,education,customer_lifetime_value,income,monthly_premium_auto,number_of_open_complaints,policy_type,vehicle_class,total_claim_amount
0,RB50392,Washington,Unknown,Master,793690,0,1000,0,Personal Auto,Four-Door Car,2.70
1,QZ44356,Arizona,F,Bachelor,697953,0,94,0,Personal Auto,Four-Door Car,"1,131.46"
2,AI49188,Nevada,F,Bachelor,1288743,48767,108,0,Personal Auto,Two-Door Car,566.47
3,WW63253,California,M,Bachelor,764586,0,106,0,Corporate Auto,SUV,529.88
4,GA49547,Washington,M,High School or Below,536307,36357,68,0,Personal Auto,Four-Door Car,17.27


(9138, 11)


Check the combined dataset

In [11]:
# check for missing values
combined_df.isna().sum()

customer                     1
state                        0
gender                       0
education                    1
customer_lifetime_value      0
income                       0
monthly_premium_auto         0
number_of_open_complaints    0
policy_type                  1
vehicle_class                1
total_claim_amount           1
dtype: int64

In [12]:
# Inspect Data Types and General info
combined_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9138 entries, 0 to 9137
Data columns (total 11 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   customer                   9137 non-null   object 
 1   state                      9138 non-null   object 
 2   gender                     9138 non-null   object 
 3   education                  9137 non-null   object 
 4   customer_lifetime_value    9138 non-null   int64  
 5   income                     9138 non-null   int64  
 6   monthly_premium_auto       9138 non-null   int64  
 7   number_of_open_complaints  9138 non-null   int64  
 8   policy_type                9137 non-null   object 
 9   vehicle_class              9137 non-null   object 
 10  total_claim_amount         9137 non-null   float64
dtypes: float64(1), int64(4), object(6)
memory usage: 785.4+ KB


In [13]:
# check for Duplicate Rows

combined_df.duplicated().sum()

np.int64(3)

In [14]:
# Remove duplicates
combined_df = combined_df.drop_duplicates()

combined_df.reset_index(drop=True, inplace=True)

In [15]:
combined_df.duplicated

<bound method DataFrame.duplicated of      customer       state   gender             education  \
0     RB50392  Washington  Unknown                Master   
1     QZ44356     Arizona        F              Bachelor   
2     AI49188      Nevada        F              Bachelor   
3     WW63253  California        M              Bachelor   
4     GA49547  Washington        M  High School or Below   
...       ...         ...      ...                   ...   
9130  LA72316  California        M              Bachelor   
9131  PK87824  California        F               College   
9132  TD14365  California        M              Bachelor   
9133  UP19263  California        M               College   
9134  Y167826  California        M               College   

      customer_lifetime_value  income  monthly_premium_auto  \
0                      793690       0                  1000   
1                      697953       0                    94   
2                     1288743   48767               

In [16]:
# statistical Summary
combined_df.describe()

,customer_lifetime_value,income,monthly_premium_auto,number_of_open_complaints,total_claim_amount
count,"9,135.00","9,135.00","9,135.00","9,135.00","9,134.00"
mean,"182,594.14","37,820.71",110.38,0.04,430.48
std,"441,207.51","30,360.15",581.44,0.31,289.62
min,"1,898.00",0.00,0.00,0.00,0.10
25%,"4,650.00",0.00,68.00,0.00,266.96
50%,"7,716.00","34,236.00",83.00,0.00,377.51
75%,"26,199.50","62,446.00",109.00,0.00,546.05
max,"5,816,655.00","99,981.00","35,354.00",5.00,"2,893.24"


In [17]:
# Save the final dataset
combined_df.to_csv(
    "combined_cleaned_insurance.csv",
    index=False
)

CONCLUSION:

We first reused the cleaning function developed in the previous lab to ensure consistency across all datasets. 

After cleaning file2 and file3, we concatenated the three datasets because they share the same structure and represent additional customer records.
* we did not use merge because all 3 files contain the same variables, and we adding more customers(row not new columns.)
* 
Finally, duplicate records were removed and the combined dataset was saved for further analysis.

# Challenge 2 — Structuring Data

We now load the marketing dataset:

`marketing_customer_analysis_clean.csv`

This dataset includes:

- Sales channel  
- Customer lifetime value  
- Response  
- Policy details  
- Vehicle details  


In [18]:
# Load the cleaned marketing dataset

marketing = pd.read_csv("marketing_customer_analysis_clean.csv")

#view the first five rows
marketing.head()


,unnamed:_0,customer,state,customer_lifetime_value,response,coverage,education,effective_to_date,employmentstatus,gender,income,location_code,marital_status,monthly_premium_auto,months_since_last_claim,months_since_policy_inception,number_of_open_complaints,number_of_policies,policy_type,policy,renew_offer_type,sales_channel,total_claim_amount,vehicle_class,vehicle_size,vehicle_type,month
0,0,DK49336,Arizona,"4,809.22",No,Basic,College,2011-02-18,Employed,M,48029,Suburban,Married,61,7.00,52,0.00,9,Corporate Auto,Corporate L3,Offer3,Agent,292.80,Four-Door Car,Medsize,A,2
1,1,KX64629,California,"2,228.53",No,Basic,College,2011-01-18,Unemployed,F,0,Suburban,Single,64,3.00,26,0.00,1,Personal Auto,Personal L3,Offer4,Call Center,744.92,Four-Door Car,Medsize,A,1
2,2,LZ68649,Washington,"14,947.92",No,Basic,Bachelor,2011-02-10,Employed,M,22139,Suburban,Single,100,34.00,31,0.00,2,Personal Auto,Personal L3,Offer3,Call Center,480.00,SUV,Medsize,A,2
3,3,XL78013,Oregon,"22,332.44",Yes,Extended,College,2011-01-11,Employed,M,49078,Suburban,Single,97,10.00,3,0.00,2,Corporate Auto,Corporate L3,Offer2,Branch,484.01,Four-Door Car,Medsize,A,1
4,4,QA50777,Oregon,"9,025.07",No,Premium,Bachelor,2011-01-17,Medical Leave,F,23675,Suburban,Married,117,15.15,31,0.38,7,Personal Auto,Personal L2,Offer1,Branch,707.93,Four-Door Car,Medsize,A,1


In [19]:
# Check the dimensions
marketing.shape

(10910, 27)

In [20]:
# Check the column names and data types
marketing.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10910 entries, 0 to 10909
Data columns (total 27 columns):
 #   Column                         Non-Null Count  Dtype  
---  ------                         --------------  -----  
 0   unnamed:_0                     10910 non-null  int64  
 1   customer                       10910 non-null  object 
 2   state                          10910 non-null  object 
 3   customer_lifetime_value        10910 non-null  float64
 4   response                       10910 non-null  object 
 5   coverage                       10910 non-null  object 
 6   education                      10910 non-null  object 
 7   effective_to_date              10910 non-null  object 
 8   employmentstatus               10910 non-null  object 
 9   gender                         10910 non-null  object 
 10  income                         10910 non-null  int64  
 11  location_code                  10910 non-null  object 
 12  marital_status                 10910 non-null 

In [21]:
# Display summary statistics

marketing.describe()

,unnamed:_0,customer_lifetime_value,income,monthly_premium_auto,months_since_last_claim,months_since_policy_inception,number_of_open_complaints,number_of_policies,total_claim_amount,month
count,"10,910.00","10,910.00","10,910.00","10,910.00","10,910.00","10,910.00","10,910.00","10,910.00","10,910.00","10,910.00"
mean,"5,454.50","8,018.24","37,536.28",93.20,15.15,48.09,0.38,2.98,434.89,1.47
std,"3,149.59","6,885.08","30,359.20",34.44,9.78,27.94,0.89,2.40,292.18,0.50
min,0.00,"1,898.01",0.00,61.00,0.00,0.00,0.00,1.00,0.10,1.00
25%,"2,727.25","4,014.45",0.00,68.00,7.00,24.00,0.00,1.00,271.08,1.00
50%,"5,454.50","5,771.15","33,813.50",83.00,15.00,48.00,0.00,2.00,382.56,1.00
75%,"8,181.75","8,992.78","62,250.75",109.00,23.00,71.00,0.38,4.00,547.20,2.00
max,"10,909.00","83,325.38","99,981.00",298.00,35.00,99.00,5.00,9.00,"2,893.24",2.00


# Pivot 1 — Total revenue by sales channel

We compute:

*Total Revenue = Monthly Premium Auto × Number of Policies**

Then pivot by sales channel.


In [22]:
# Create a pivot table showing total revenue by sales channel
marketing["total_revenue"] = (
    marketing["monthly_premium_auto"] * marketing["number_of_policies"]
)

pivot_revenue = pd.pivot_table(
    marketing,
    values="total_revenue",
    index="sales_channel",
    aggfunc="sum"
).round(2)

pivot_revenue


,total_revenue
sales_channel,
Agent,1130616
Branch,818951
Call Center,589902
Web,471197


In [23]:
# EXERCISE 2:
# Average Customer Lifetime Value by gender and education
# Round values to two decimal places
clv_table = pd.pivot_table(
    marketing,
    index="education",
    columns="gender",
    values="customer_lifetime_value",
    aggfunc="mean"
)

# Round to two decimal places
clv_table = clv_table.round(2)

# Display the table
clv_table

gender,F,M
education,,
Bachelor,"7,874.27","7,703.60"
College,"7,748.82","8,052.46"
Doctor,"7,328.51","7,415.33"
High School or Below,"8,675.22","8,149.69"
Master,"8,157.05","8,168.83"


## Bonus — Complaints by month and policy type (long format)

We extract month from `effective_to_date` and count complaints.


In [24]:
# Count complaints by policy type and month

marketing["effective_to_date"] = pd.to_datetime(marketing["effective_to_date"])
marketing["month"] = marketing["effective_to_date"].dt.month_name()

complaints_long = marketing.groupby(
    ["policy_type", "month"]
)["number_of_open_complaints"].sum().reset_index()

complaints_long.head()


,policy_type,month,number_of_open_complaints
0,Corporate Auto,February,385.21
1,Corporate Auto,January,443.43
2,Personal Auto,February,"1,453.68"
3,Personal Auto,January,"1,727.61"
4,Special Auto,February,95.23


In [25]:
# Conclusion

In this notebook, we:

### ✔ Cleaned file2 and file3  
### ✔ Combined all three datasets  
### ✔ Loaded the marketing dataset  
### ✔ Created pivot tables  
### ✔ Generated long-format complaint tables  
### ✔ Completed both Challenge 1 and Challenge 2  

This notebook is ready for submission.


SyntaxError: invalid syntax (3568774380.py, line 3)